In [ ]:
from cro_rmi_improvement_feature.network_analyzer.notebook.gen_cluster_mitigationplan_langchain_version import *

# ==================== TEST CODE =====================
dir_path = os.getcwd()
pcg_data_path = f"{dir_path}/../data/graph/graph_data_library_no_embedding_dict.pkl"
# pcg_data_path = "/home/thanatorn/coding/cro_rmi_improvement_feature/src/cro_rmi_improvement_feature/network_analyzer/data/graph/graph_data_library_no_embedding_dict.pkl"
all_data_df, clusters, G, nodes = find_graph_properties_newpickle(
    pcg_data_path, "PCG|embedding_risk_desc_catalog|oneway_run"
)

# all_data_df, clusters, G, nodes = find_graph_properties(data_path_dict)

risk_high = all_data_df[
    all_data_df["risk_level"] >= 3
]  # TODO: need to handle the edge case when there's no high/critical risk level
# TopN risks with a given property (central/source/sink)
N_TOP = 3
risk_central = top_n_with_row_limit(
    all_data_df, "betweenness_centrality_non_weight", n=N_TOP
)
risk_source = top_n_with_row_limit(all_data_df, "out_degree", n=N_TOP)

# ======= cluster by type (high/central/source risk) =======

# cluster(s) that contains risk_high
highrisk_clusters = find_sublists_with_any(risk_high["risk_id"].tolist(), clusters)
# print(highrisk_clusters)

# cluster(s) that contains risk_central
centralrisk_clusters = find_sublists_with_any(
    risk_central["risk_id"].tolist(), clusters
)
# print(centralrisk_clusters)

# cluster(s) that contains risk_source
sourcerisk_clusters = find_sublists_with_any(risk_source["risk_id"].tolist(), clusters)
# print(sourcerisk_clusters)

# ======= ALL important clusters (contains high/central/source risks) =======
combined_list = list(
    set(
        risk_high["risk_id"].tolist()
        + risk_central["risk_id"].tolist()
        + risk_source["risk_id"].tolist()
    )
)
important_clusters = find_sublists_with_any(combined_list, clusters)

# # === STEP 5: OUTPUT PROMPTS ===
# for i, cluster in enumerate(important_clusters, 1):
#     print(f"\n--- Cluster {i} Prompt ---\n")
#     print(generate_prompt_nointro(cluster, G, nodes))

# print(generate_prompt_nointro(important_clusters[0], G, nodes))

In [8]:
# from dotenv import load_dotenv
# import os
# from typing import List, Tuple
from langchain_community.callbacks import get_openai_callback

# TODO: need to acquire the api_key from your directory
# load_dotenv("../../.env")

# estimate_cost.total_cost_THB = 0  # initialization
usage_count_list = []

# notice, this is a single cluster
cluster_risk_to_plan = generate_prompt_nointro(important_clusters[1], G, nodes)
# cluster_risk_to_plan = generate_prompt_nointro(highrisk_clusters, G, nodes)

# gen 'cluster' mitigation plans
# response = get_response_control_cluster(cluster_risk_to_plan)  # accept single cluster
with get_openai_callback() as cb:
    response = get_response_control_cluster(
        cluster_risk_to_plan,
        # model="gpt-4o-mini",
        model="gpt-4.1",
    )  # accept single cluster
    print(cb)
    thb = cb.total_cost * 35
    print(f"total cost (THB): {thb}")

# response = get_response_test(cluster_risk_to_plan)

# usage_count = response.usage
# usage_count_list.append(usage_count)
# MODEL = "gpt-4.1"  # "gpt-4o"
# estimate_cost(res_usage=usage_count_list, type="multi", model=MODEL)

Tokens Used: 3201
	Prompt Tokens: 1348
		Prompt Tokens Cached: 0
	Completion Tokens: 1853
		Reasoning Tokens: 0
Successful Requests: 1
Total Cost (USD): $0.0
total cost (THB): 0.0


In [9]:
cb.__dict__

{'_lock': <unlocked _thread.lock object at 0x14feb8a80>,
 'total_cost': 0.0,
 'total_tokens': 3201,
 'prompt_tokens': 1348,
 'prompt_tokens_cached': 0,
 'completion_tokens': 1853,
 'reasoning_tokens': 0,
 'successful_requests': 1}

In [10]:
usd_out = cb.completion_tokens * 8.00 / 1_000_000
usd_in = cb.prompt_tokens * 2.00 / 1_000_000

thb_out = usd_out * 35
thb_in = usd_in * 35

print(f"usd_out: {usd_out}")
print(f"usd_in: {usd_in}")
print(f"usd_total: {usd_out + usd_in}")
print(f"thb_out: {thb_out}")
print(f"thb_in: {thb_in}")
print(f"thb_total: {thb_out + thb_in}")

usd_out: 0.014824
usd_in: 0.002696
usd_total: 0.01752
thb_out: 0.51884
thb_in: 0.09436
thb_total: 0.6132


In [4]:
response_dict = response.model_dump()
response_dict

{'detailed_action_plan': [{'cluster_plan_name': 'Integrated Service Excellence Program',
   'objective': 'To systematically improve service quality, reduce delivery errors, and increase customer satisfaction, thus dampening the propagation of dissatisfaction and uncompetitive service through the risk network.',
   'priority_level': 'High',
   'key_impact': 'By enhancing frontline service processes and quality controls, this program addresses root causes of dissatisfaction and poor quality, preventing their escalation into broader issues like uncompetitive services and market share loss.',
   'risks_addressed': [{'risk_addressed': 'risk_20250513_41: Poor service quality',
     'target_risk_likelihood': 'Unlikely',
     'target_risk_impact': 'Minor'},
    {'risk_addressed': 'risk_20250513_54: Service-related dissatisfaction',
     'target_risk_likelihood': 'Unlikely',
     'target_risk_impact': 'Minor'},
    {'risk_addressed': 'risk_20250513_65: Wrong delivery',
     'target_risk_likelih

In [ ]:
# save to json
tmp_res = response.choices[0].message.content
report_json = json.loads(tmp_res)

data = []
data.append(report_json)

# input_file = report_json_folder + str(selected_year) + '_Q' + str(selected_quarter) + '_' + selected_company_report + '_json_riskcontrol.json'
file_path = f"{dir_path}/clustercontrol.json"

with open(file_path, "w") as file:
    json.dump(data, file, indent=4, ensure_ascii=False)

report_json

AttributeError: 'RiskMitigationActionClusterPlan' object has no attribute 'choices'